# HumAID — Zero-shot Classification (Filtered Labels, Batch API, Sharding, Stand Alone)

- **Filtered labels (per event):** prompts + JSON schema only list labels that appear in that event’s ground truth → reduces out-of-scope (OOS) predictions.
- **Batch API flow:** build `requests.jsonl` → upload → create batch → poll → download `outputs.jsonl` (and `errors.jsonl` if any).
- **Patch pass:** after batch completes, any missing/blank predictions are re-classified synchronously so `predictions.csv` has one row per input.
- **Stratified sharding (optional):** split large events into *k* shards **preserving class ratios**; use the **same** event-level labels + rules for all shards; merge predictions back in original order.
- **Reporting:** confusion matrices (counts + row-normalized), per-class F1/error, mistakes CSV, and a sortable `results/index.html`.  
  - **Scope** = label universe used for metrics (default `truth`).  
  - **OOS preds** = predictions not in the truth set (QA signal).

## Key settings
- `MODEL` (e.g., `gpt-4o`), `RULES` (e.g., `RULES_1`), `TAG`
- `DRYRUN_N`, `POLL_SECS`
- Token budgeting: `BATCH_TOKEN_LIMIT`, `SAFETY_MARGIN`, `MAX_OUTPUT_TOKENS`
- `.env` with `OPENAI_API_KEY_1` (and optionally a second key)

# 0) Setup

In [1]:
# test_gpt5_single_event.py  — stand-alone probe for a single TSV with GPT-5 (Responses API)
# Usage (script):  python test_gpt5_single_event.py --tsv Dataset/HumAID/event_x/event_x_test.tsv --model gpt-5-mini
# In a notebook: just run the cell; set TSV_PATH below.

import os, json, re, time, argparse, uuid, requests
import pandas as pd

# --- folder planning (match package layout) ----------------------------------
from datetime import datetime
import re
from pathlib import Path
from dotenv import load_dotenv; load_dotenv()
from rules import RULES_1

SYSTEM_PROMPT = (
  "You are a precise tweet classifier for humanitarian-response content.\n"
  "Choose exactly one label from the allowed labels.\n"
  "If unrelated to humanitarian contexts, choose 'not_humanitarian'.\n"
  "Never invent labels not listed in the allowed labels.\n"
  "Output JSON matching the provided schema; no extra fields."
)

HUMAID_LABELS = [
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "infrastructure_and_utility_damage",
    "injured_or_dead_people",
    "missing_or_found_people",
    "requests_or_urgent_needs",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
    "other_relevant_information",
    "not_humanitarian",
]

def filter_labels_in_scope(labels: list[str]) -> list[str]:
    """Preserve incoming order, keep only known HumAID labels, and dedupe."""
    seen = set()
    out = []
    for l in labels:
        if l in HUMAID_LABELS and l not in seen:
            seen.add(l)
            out.append(l)
    return out

# Keep canonical ordering if desired
def order_by_humaid(labels: list[str]) -> list[str]:
    return [l for l in HUMAID_LABELS if l in labels]

def parse_rules_kv(rules_text: str) -> dict[str, str]:
    """
    Parse compact one-liners like:
      - caution_and_advice: warnings/instructions/tips
    Returns a map: {label -> ORIGINAL LINE (including leading "- ")}.
    """
    kv: dict[str, str] = {}
    for m in re.finditer(r'^\s*-\s*([a-z0-9_]+)\s*:\s*(.+?)\s*$',
                         rules_text, flags=re.I | re.M):
        label = m.group(1).strip()
        whole_line = m.group(0).rstrip()
        kv[label] = whole_line
    return kv    
    
def parse_rules_blocks(rules_text: str) -> dict[str, str]:
    """
    Expects blocks in the format:
    - label_name
      Definition: ...
      Include: ...
      Exclude: ...
    """
    blocks = {}
    # Split on lines that start with "- <label>"
    parts = re.split(r'\n(?=-\s+[a-z0-9_]+)', "\n"+rules_text.strip(), flags=re.I)
    for p in parts:
        m = re.match(r'-\s+([a-z0-9_]+)\s*\n(.+)$', p.strip(), flags=re.I|re.S)
        if m:
            label = m.group(1).strip()
            body  = m.group(2).rstrip()
            blocks[label] = f"- {label}\n{body}\n"
    return blocks

def slice_rules_for_labels(rules_text: str, labels: list[str]) -> str:
    """
    Return ORIGINAL rule text limited to `labels`.
    - If rules are compact one-liners, emit the ORIGINAL lines (no edits).
    - If rules are multi-line blocks, emit the ORIGINAL blocks.
    - Order is HumAID canonical (labels in HUMAID_LABELS that appear in `labels`).
    - If neither parser matches, return the original rules text trimmed.
    """
    labels_ord = order_by_humaid(labels)

    # 1) Try compact one-liners
    kv = parse_rules_kv(rules_text)
    if kv:
        lines = [kv[l] for l in labels_ord if l in kv]
        return "\n".join(lines).strip()

    # 2) Fallback to multi-line blocks
    blocks = parse_rules_blocks(rules_text)
    kept = [blocks[l].rstrip() for l in labels_ord if l in blocks]
    if kept:
        return "\n\n".join(kept).strip()

    # 3) Last resort: unchanged, trimmed
    return rules_text.strip()

def _infer_event_split(tsv_path: str) -> tuple[str, str]:
    p = Path(tsv_path)
    event = p.parent.name                       # e.g., kerala_floods_2018
    m = re.search(r'_(train|dev|test)\.tsv$', p.name, flags=re.I)
    split = m.group(1).lower() if m else "unknown"
    return event, split

def plan_dirs_like_package(tsv_path: str, out_root: str, model: str, tag: str):
    event, split = _infer_event_split(tsv_path)
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = Path(out_root) / event / split / model / f"{stamp}-{tag}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return {
        "dir": run_dir,
        "predictions_csv": run_dir / "predictions.csv",
        "summary_json":    run_dir / "summary.json",
        "meta_json":       run_dir / "meta.json",
    }


# ---------- Config ----------
OPENAI_BASE = "https://api.openai.com/v1"
API_KEY = os.getenv("OPENAI_API_KEY_1") or os.getenv("OPENAI_API_KEY")
assert API_KEY, "Set OPENAI_API_KEY_1 or OPENAI_API_KEY in your env."
HEAD = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

# Retry configuration (can be modified as needed)
MAX_RETRY_ATTEMPTS = 5
RETRY_DELAY = 1.0

# ---------- IO ----------
def load_tsv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    # Normalize required columns
    if "tweet_id" not in df.columns or "tweet_text" not in df.columns:
        raise ValueError("TSV must contain 'tweet_id' and 'tweet_text'.")
    # Optional ground truth
    if "class_label" not in df.columns:
        df["class_label"] = ""
    df["tweet_id"] = df["tweet_id"].astype(str)
    df["tweet_text"] = df["tweet_text"].astype(str)
    df["class_label"] = df["class_label"].astype(str)
    return df

def event_labels(df: pd.DataFrame, supplied: list[str] | None = None) -> list[str]:
    if supplied:
        return order_by_humaid(filter_labels_in_scope(list(supplied)))

    seen = set()
    present = []
    for lab in df["class_label"].astype(str):
        lab = lab.strip()
        if lab and lab.lower() not in {"nan", "none"} and lab not in seen:
            seen.add(lab)
            present.append(lab)

    if not present:
        return HUMAID_LABELS.copy()

    return order_by_humaid(filter_labels_in_scope(present))

# ---------- Schema / parsing ----------
def make_schema(labels: list[str], keep_conf: bool=False) -> dict:
    # GPT-5 works best with a minimal schema. Confidence optional and may be ignored.
    props = {"label": {"type": "string", "enum": labels}}
    if keep_conf:
        props["confidence"] = {"type": "number", "minimum": 0, "maximum": 1}
    return {
        "type": "object",
        "properties": props,
        "required": ["label"],
        "additionalProperties": False,
    }

def normalize_label(text: str, labels: list[str]) -> str | None:
    if not text:
        return None
    t = text.strip().strip('"\'')

    # Exact (case-insensitive)
    for L in labels:
        if t.lower() == L.lower():
            return L

    # Try JSON fragment
    m = re.search(r'"label"\s*:\s*"([^"]+)"', t)
    if m:
        cand = m.group(1).strip()
        for L in labels:
            if cand.lower() == L.lower():
                return L

    # Try loose "label: X"
    m = re.search(r'\blabel\s*:\s*([A-Za-z0-9\-\_]+)', t)
    if m:
        cand = m.group(1).strip()
        for L in labels:
            if cand.lower() == L.lower():
                return L

    # Fuzzy contains (last resort)
    for L in labels:
        if L.lower() in t.lower():
            return L
    return None

def extract_from_responses(resp_json: dict, labels: list[str]) -> tuple[str | None, dict]:
    # Preferred: output_parsed (Structured Outputs on Responses API)
    op = resp_json.get("output_parsed")
    if isinstance(op, list) and op:
        obj = op[0]
        if isinstance(obj, dict) and "label" in obj:
            return str(obj["label"]).strip(), obj

    # Fallback: the rendered message text (avoid "reasoning" items)
    text = ""
    out = resp_json.get("output", [])
    try:
        for item in out:
            if item.get("type") == "message":
                # content is a list of fragments
                parts = item.get("content", [])
                if parts and isinstance(parts, list):
                    # choose first text fragment
                    for p in parts:
                        if isinstance(p, dict) and "text" in p:
                            text = p["text"]
                            break
                if text:
                    break
        if not text and out:
            parts = out[0].get("content", [])
            if parts and isinstance(parts, list):
                for p in parts:
                    if isinstance(p, dict) and "text" in p:
                        text = p["text"]
                        break
    except Exception:
        text = ""

    # Try to parse JSON
    if text:
        try:
            obj = json.loads(text)
            if isinstance(obj, dict) and "label" in obj:
                return str(obj["label"]).strip(), obj
        except Exception:
            pass

    lab = normalize_label(text, labels)
    return lab, ({"label": lab} if lab else {})

# ---------- API (Responses) with RETRY ----------
def responses_classify_row(model: str, text: str, labels: list[str], rules: str, 
                          max_output_tokens: int = 5000,
                          timeout: int = 180, poll_retries: int = 30, poll_delay: float = 3.0) -> tuple[str | None, dict]:
    """
    Call /responses for a single row WITH AUTOMATIC RETRY for invalid labels.
    Returns (label, parsed_dict)
    """
    
    for attempt in range(MAX_RETRY_ATTEMPTS):
        try:
            schema = make_schema(labels, keep_conf=False)
            
            # Make prompt more emphatic on retries
            if attempt == 0:
                user_prompt = (
                    "Classify this tweet into exactly ONE of the allowed labels.\n"
                    f"Allowed labels (enum): {', '.join(labels)}\n\n"
                    "Rules:\n"
                    f"{rules}\n\n" 
                    f'Tweet: """{text.strip()}"""'  
                )
            else:
                # Stronger emphasis on retry
                user_prompt = (
                    "IMPORTANT: You MUST choose EXACTLY ONE label from this list:\n"
                    f"[{', '.join(labels)}]\n\n"
                    "Do NOT make up labels. ONLY use the labels provided above.\n\n"
                    "Rules:\n"
                    f"{rules}\n\n" 
                    f'Tweet: """{text.strip()}"""'  
                )
            
            body = {
                "model": model,
                "input": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_prompt},
                ],
                "text": {
                    "format": {
                        "type": "json_schema",
                        "name": "tweet_label",
                        "schema": schema,
                        "strict": True
                    }
                },
                "max_output_tokens": max_output_tokens
            }

            r = requests.post(f"{OPENAI_BASE}/responses", headers=HEAD, json=body, timeout=timeout)
            
            if r.status_code != 200:
                if attempt < MAX_RETRY_ATTEMPTS - 1:
                    print(f"  ⚠ HTTP {r.status_code} on attempt {attempt + 1}/{MAX_RETRY_ATTEMPTS}, retrying...")
                    time.sleep(RETRY_DELAY)
                    continue
                return None, {"error": f"HTTP {r.status_code}: {r.text[:200]}", "attempts": attempt + 1}

            j = r.json()
            status = j.get("status")
            
            if status == "incomplete":
                rid = j.get("id")
                for _ in range(poll_retries):
                    time.sleep(poll_delay)
                    rr = requests.get(f"{OPENAI_BASE}/responses/{rid}", 
                                    headers={"Authorization": f"Bearer {API_KEY}"}, 
                                    timeout=timeout)
                    if rr.status_code != 200:
                        continue
                    j = rr.json()
                    if j.get("status") == "completed":
                        break
                    if j.get("status") == "failed":
                        if attempt < MAX_RETRY_ATTEMPTS - 1:
                            print(f"  ⚠ Response failed on attempt {attempt + 1}/{MAX_RETRY_ATTEMPTS}, retrying...")
                            time.sleep(RETRY_DELAY)
                            break
                        return None, {"error": f"Response failed: {j.get('error')}", "attempts": attempt + 1}

            label, parsed = extract_from_responses(j, labels)
            
            # Validate the label
            if label and label in labels:
                if attempt > 0:
                    print(f"  ✓ Got valid label '{label}' on attempt {attempt + 1}/{MAX_RETRY_ATTEMPTS}")
                return label, parsed
            
            # Invalid label, retry if we have attempts left
            if attempt < MAX_RETRY_ATTEMPTS - 1:
                print(f"  ⚠ Invalid label '{label}' on attempt {attempt + 1}/{MAX_RETRY_ATTEMPTS}, retrying...")
                time.sleep(RETRY_DELAY)
            else:
                print(f"  ✗ Failed to get valid label after {MAX_RETRY_ATTEMPTS} attempts (got: '{label}')")
                # Use fallback to not_humanitarian if available
                if "not_humanitarian" in labels:
                    print(f"    → Using fallback: 'not_humanitarian'")
                    return "not_humanitarian", {"fallback": True, "original": label, "attempts": MAX_RETRY_ATTEMPTS}
                return None, {"error": f"Invalid label after {MAX_RETRY_ATTEMPTS} attempts", "last_label": label, "attempts": MAX_RETRY_ATTEMPTS}
                
        except Exception as e:
            if attempt < MAX_RETRY_ATTEMPTS - 1:
                print(f"  ⚠ Exception on attempt {attempt + 1}/{MAX_RETRY_ATTEMPTS}: {e}, retrying...")
                time.sleep(RETRY_DELAY)
            else:
                return None, {"error": f"Exception: {str(e)}", "attempts": attempt + 1}
    
    return None, {"error": f"Max attempts ({MAX_RETRY_ATTEMPTS}) reached"}

# ---------- Metrics ----------
def compute_metrics(df: pd.DataFrame, col_truth="class_label", col_pred="predicted_label") -> dict:
    """
    Compute metrics properly, handling NaN values like eval.py does.
    """
    try:
        # Try to import from eval.py first
        from humaidclf.eval import macro_f1
        
        m_f1 = macro_f1(df, truth_col=col_truth, pred_col=col_pred, scope="truth")
        
        # Also compute additional metrics using sklearn
        from sklearn.metrics import f1_score, classification_report
        
        # Clean data like eval.py does
        df_clean = df.copy()
        df_clean[col_truth] = df_clean[col_truth].astype(str).str.strip()
        df_clean[col_pred] = df_clean[col_pred].astype(str).str.strip()
        df_clean[col_truth] = df_clean[col_truth].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df_clean[col_pred] = df_clean[col_pred].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df_clean = df_clean.dropna(subset=[col_truth, col_pred])
        
        if not df_clean.empty:
            y_true = df_clean[col_truth].tolist()
            y_pred = df_clean[col_pred].tolist()
            truth_labels = sorted(set(y_true))
            
            weighted = f1_score(y_true, y_pred, labels=truth_labels, average="weighted", zero_division=0)
            rep = classification_report(y_true, y_pred, labels=truth_labels, output_dict=True, zero_division=0)
            
            return {
                "macro_f1": float(m_f1) if pd.notna(m_f1) else 0.0,
                "weighted_f1": weighted,
                "per_class": rep,
                "num_evaluated": len(df_clean),
                "num_invalid": len(df) - len(df_clean)
            }
        else:
            return {"macro_f1": 0.0, "weighted_f1": 0.0, "num_evaluated": 0, "num_invalid": len(df)}
            
    except ImportError:
        # Fallback if eval.py is not available
        from sklearn.metrics import f1_score, classification_report
        
        # Clean data
        df_clean = df.copy()
        for col in [col_truth, col_pred]:
            df_clean[col] = df_clean[col].astype(str).str.strip()
            df_clean[col] = df_clean[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df_clean = df_clean.dropna(subset=[col_truth, col_pred])
        
        if df_clean.empty:
            return {"macro_f1": 0.0, "weighted_f1": 0.0, "num_evaluated": 0}
            
        y_true = df_clean[col_truth].tolist()
        y_pred = df_clean[col_pred].tolist()
        truth_labels = sorted(set(y_true))
        
        macro = f1_score(y_true, y_pred, labels=truth_labels, average="macro", zero_division=0)
        weighted = f1_score(y_true, y_pred, labels=truth_labels, average="weighted", zero_division=0)
        rep = classification_report(y_true, y_pred, labels=truth_labels, output_dict=True, zero_division=0)
        
        return {
            "macro_f1": macro, 
            "weighted_f1": weighted, 
            "per_class": rep,
            "num_evaluated": len(df_clean),
            "num_invalid": len(df) - len(df_clean)
        }
    except Exception as e:
        return {"error": str(e)}

# ---------- Main single-event runner ----------
def run_single_event(tsv_path: str,
                     model: str = "gpt-5-mini",
                     rules: str = "",
                     out_root: str = "runs",
                     tag: str = "responses-gpt5-probe",
                     max_output_tokens: int = 5000,
                     max_rows: int | None = None):
    """
    Run single event classification with automatic retry on invalid labels.
    Keeps the same interface as before but with improved robustness.
    """
    df = load_tsv(tsv_path)
    if max_rows:
        df = df.head(max_rows).copy()

    plan = plan_dirs_like_package(tsv_path, out_root=out_root, model=model, tag=tag)

    labels = event_labels(df)
    print(f"Event: {Path(tsv_path).parent.name}")
    print(f"Labels in scope: {labels}")

    # Default rules if none provided
    if not rules:
        try:
            from rules import RULES_1 as _R 
        except ImportError:
            from rules import RULES_2 as _R
        rules = _R

    # Slice rules to only the labels in this event schema
    rules_scoped = slice_rules_for_labels(rules, labels)        
    
    print(f"Processing {len(df)} rows with up to {MAX_RETRY_ATTEMPTS} attempts per row...")

    # Track statistics
    stats = {
        "total_rows": len(df),
        "successful_first_try": 0,
        "successful_with_retry": 0,
        "used_fallback": 0,
        "failed_completely": 0,
    }

    # single-label fast path
    if len(labels) == 1:
        only = labels[0]
        out = df[["tweet_id","tweet_text","class_label"]].copy()
        out["predicted_label"] = only
        out["confidence"] = 1.0
        out.to_csv(plan["predictions_csv"], index=False)
        metrics = compute_metrics(out)
        metrics["mode"] = "single-label-fast-path"
        with open(plan["summary_json"], "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)
        with open(plan["meta_json"], "w", encoding="utf-8") as f:
            json.dump({"model": model, "tag": tag, "tsv": str(tsv_path)}, f, indent=2)
        print(f"Single label detected, using fast path: {only}")
        return out, metrics

    # multi-label path with retry logic
    rows = []
    for i, r in df.iterrows():
        lab, parsed = responses_classify_row(
            model, r["tweet_text"], labels, rules_scoped,
            max_output_tokens=max_output_tokens, timeout=180,
            poll_retries=30, poll_delay=3.0
        )
        
        # Track statistics
        attempts = parsed.get("attempts", 1) if isinstance(parsed, dict) else 1
        if lab and lab in labels:
            if parsed.get("fallback"):
                stats["used_fallback"] += 1
            elif attempts == 1:
                stats["successful_first_try"] += 1
            else:
                stats["successful_with_retry"] += 1
        else:
            stats["failed_completely"] += 1
            lab = pd.NA  # Use pandas NA instead of None or empty string
        
        rows.append({
            "tweet_id": r["tweet_id"],
            "tweet_text": r["tweet_text"],
            "class_label": r["class_label"],
            "predicted_label": lab if pd.notna(lab) else pd.NA,
            "confidence": parsed.get("confidence") if isinstance(parsed, dict) else None,
        })
        
        if (i+1) % 20 == 0:
            success = stats["successful_first_try"] + stats["successful_with_retry"] + stats["used_fallback"]
            rate = success / (i+1)
            print(f"Processed {i+1}/{len(df)} rows... (success rate: {rate:.1%})")

    out = pd.DataFrame(rows)
    out.to_csv(plan["predictions_csv"], index=False)

    metrics = compute_metrics(out)
    metrics["classification_stats"] = stats
    
    with open(plan["summary_json"], "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    
    with open(plan["meta_json"], "w", encoding="utf-8") as f:
        json.dump({
            "model": model, 
            "tag": tag, 
            "tsv": str(tsv_path),
            "max_retry_attempts": MAX_RETRY_ATTEMPTS,
            "retry_delay": RETRY_DELAY
        }, f, indent=2)

    print("\n" + "="*50)
    print("CLASSIFICATION COMPLETE")
    print("="*50)
    print(f"Total rows: {stats['total_rows']}")
    print(f"Success on first try: {stats['successful_first_try']} ({stats['successful_first_try']/stats['total_rows']:.1%})")
    print(f"Success with retry: {stats['successful_with_retry']} ({stats['successful_with_retry']/stats['total_rows']:.1%})")
    print(f"Used fallback: {stats['used_fallback']} ({stats['used_fallback']/stats['total_rows']:.1%})")
    print(f"Failed completely: {stats['failed_completely']} ({stats['failed_completely']/stats['total_rows']:.1%})")
    
    if metrics.get("macro_f1") is not None:
        print(f"\nMacro F1: {metrics['macro_f1']:.4f}")
        if metrics.get("weighted_f1") is not None:
            print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
        if metrics.get("num_invalid"):
            print(f"Invalid predictions excluded: {metrics['num_invalid']}")
    
    print(f"\nResults saved to: {plan['predictions_csv']}")
    return out, metrics


# ---------- Script entry ----------
# if __name__ == "__main__":
#     p = argparse.ArgumentParser()
#     p.add_argument("--tsv", required=True, help="Path to a single event TSV (tweet_id, tweet_text, optional class_label).")
#     p.add_argument("--model", default="gpt-5-mini", help="Model name, e.g., gpt-5-mini or gpt-5.")
#     p.add_argument("--rules", default="", help="Rules text to include in the prompt.")
#     p.add_argument("--out_root", default="runs", help="Root directory for outputs.")
#     p.add_argument("--tag", default="responses-gpt5-probe", help="Tag for the run folder.")
#     p.add_argument("--max_rows", type=int, default=None, help="Limit rows for a quick test.")
#     args = p.parse_args()
    
#     # Load rules if needed
#     rules_text = args.rules
#     if args.rules == "RULES_1":
#         from rules import RULES_1
#         rules_text = RULES_1
    
#     run_single_event(
#         args.tsv, 
#         model=args.model, 
#         rules=rules_text, 
#         out_root=args.out_root,
#         tag=args.tag,
#         max_rows=args.max_rows
#     )

# Test code

In [2]:
# === DRY-RUN: inspect labels, rules, schema, and prompts (no API call) ===

# --- Config for the test ---
TSV_PATH   = "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv"  # <-- change to your TSV
N_SAMPLES  = 3                                          # number of tweets to preview
MODEL_NAME = "gpt-5-mini"                               # just for display

print("=== 1) Load TSV ===")
df = load_tsv(TSV_PATH)
print(f"Loaded {len(df)} rows from: {TSV_PATH}")

# --- labels in scope (truth-only scope, like runner) ---
print("\n=== 2) Event labels ===")
labels = event_labels(df)
print("Labels in scope:", labels)

# --- rules (scoped to those labels) ---
print("\n=== 3) Scoped rules from RULES_1 ===")
rules_full   = RULES_1
rules_scoped = slice_rules_for_labels(rules_full, labels)
print(rules_scoped)

# --- JSON schema that we pass to the Responses API ---
print("\n=== 4) JSON schema (make_schema) ===")
schema = make_schema(labels, keep_conf=False)
print(json.dumps(schema, indent=2))

# --- system prompt (as used in responses_classify_row) ---
print("\n=== 5) SYSTEM_PROMPT ===")
print(SYSTEM_PROMPT)

# --- user prompts for a few sample tweets (attempt == 0 style) ---
print("\n=== 6) USER prompts for first few tweets (no API call) ===")
sample_df = df.head(N_SAMPLES).copy()

for i, row in sample_df.iterrows():
    tweet_text = row["tweet_text"]
    tweet_id   = row["tweet_id"]

    user_prompt = (
        "Classify this tweet into exactly ONE of the allowed labels.\n"
        f"Allowed labels (enum): {', '.join(labels)}\n\n"
        "Rules:\n"
        f"{rules_scoped}\n\n"
        f'Tweet: """{tweet_text.strip()}"""'
    )

    print("\n" + "-" * 80)
    print(f"Sample #{i}  |  tweet_id={tweet_id}  |  model={MODEL_NAME}")
    print("-" * 80)
    print(user_prompt)

# (optional) the full JSON body we would POST:
first_row = sample_df.iloc[0]
test_body = {
    "model": MODEL_NAME,
    "input": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},  # from the last loop iteration
    ],
    "text": {
        "format": {
            "type": "json_schema",
            "name": "tweet_label",
            "schema": schema,
            "strict": True,
        }
    },
    "max_output_tokens": 40,
}
print("\n=== 7) Example request body (no network call) ===")
print(json.dumps(test_body, indent=2))


=== 1) Load TSV ===
Loaded 435 rows from: Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv

=== 2) Event labels ===
Labels in scope: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort', 'sympathy_and_support', 'other_relevant_information', 'not_humanitarian']

=== 3) Scoped rules from RULES_1 ===
- caution_and_advice: warnings/instructions/tips
- displaced_people_and_evacuations: evacuations, relocation, shelters
- infrastructure_and_utility_damage: damage/outages to roads/bridges/power/water/buildings
- injured_or_dead_people: injuries, casualties, fatalities
- requests_or_urgent_needs: asking for help/supplies/SOS
- rescue_volunteering_or_donation_effort: offering help, donation, organizing aid
- sympathy_and_support: prayers/condolences, no actionable info
- other_relevant_information: on-topic but none of the above
- not

# Experiment

In [3]:
from rules import RULES_1

out, metrics = run_single_event(
    "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv",
    model="gpt-5-nano",
    rules=RULES_1,
    out_root="runs",
    tag="modeR-gpt-5-nano-RULES1-probe",
    max_output_tokens=500,
    max_rows=None
)

Event: kaikoura_earthquake_2016
Labels in scope: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort', 'sympathy_and_support', 'other_relevant_information', 'not_humanitarian']
Processing 435 rows with up to 5 attempts per row...
  ⚠ Invalid label 'None' on attempt 1/5, retrying...
  ⚠ Invalid label 'None' on attempt 2/5, retrying...
  ✓ Got valid label 'other_relevant_information' on attempt 3/5
  ⚠ Invalid label 'None' on attempt 1/5, retrying...
  ⚠ Invalid label 'None' on attempt 2/5, retrying...
  ✓ Got valid label 'infrastructure_and_utility_damage' on attempt 3/5
Processed 20/435 rows... (success rate: 100.0%)
  ⚠ Invalid label 'None' on attempt 1/5, retrying...
  ⚠ Invalid label 'None' on attempt 2/5, retrying...
  ✓ Got valid label 'caution_and_advice' on attempt 3/5
Processed 40/435 rows... (success rate: 100.0%)
  ⚠ Invalid label 'None' on

In [2]:
from rules import RULES_1

out, metrics = run_single_event(
    "Dataset/HumAID/canada_wildfires_2016/canada_wildfires_2016_test.tsv",
    model="gpt-5-nano",
    rules=RULES_1,
    out_root="runs",
    tag="modeR-gpt-5-nano-RULES1-probe",
    max_rows=None
)

Event: canada_wildfires_2016
Labels in scope: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort', 'sympathy_and_support', 'other_relevant_information', 'not_humanitarian']
Processing 445 rows with up to 5 attempts per row...
Processed 20/445 rows... (success rate: 100.0%)
Processed 40/445 rows... (success rate: 100.0%)
Processed 60/445 rows... (success rate: 100.0%)
Processed 80/445 rows... (success rate: 100.0%)
Processed 100/445 rows... (success rate: 100.0%)
Processed 120/445 rows... (success rate: 100.0%)
Processed 140/445 rows... (success rate: 100.0%)
Processed 160/445 rows... (success rate: 100.0%)
Processed 180/445 rows... (success rate: 100.0%)
Processed 200/445 rows... (success rate: 100.0%)
Processed 220/445 rows... (success rate: 100.0%)
Processed 240/445 rows... (success rate: 100.0%)
Processed 260/445 rows... (success rate: 100.0%)
Processed 280/445 rows... (s

In [2]:
from rules import RULES_1

out, metrics = run_single_event(
    "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv",
    model="gpt-5-nano",
    rules=RULES_1,
    out_root="runs",
    tag="modeR-gpt-5-nano-RULES1-probe",
    max_output_tokens=2000,
    max_rows=None
)

Event: kaikoura_earthquake_2016
Labels in scope: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort', 'sympathy_and_support', 'other_relevant_information', 'not_humanitarian']
Processing 435 rows with up to 5 attempts per row...
Processed 20/435 rows... (success rate: 100.0%)
Processed 40/435 rows... (success rate: 100.0%)
Processed 60/435 rows... (success rate: 100.0%)
Processed 80/435 rows... (success rate: 100.0%)
Processed 100/435 rows... (success rate: 100.0%)
Processed 120/435 rows... (success rate: 100.0%)
Processed 140/435 rows... (success rate: 100.0%)
Processed 160/435 rows... (success rate: 100.0%)
Processed 180/435 rows... (success rate: 100.0%)
Processed 200/435 rows... (success rate: 100.0%)
Processed 220/435 rows... (success rate: 100.0%)
Processed 240/435 rows... (success rate: 100.0%)
Processed 260/435 rows... (success rate: 100.0%)

In [2]:
from rules import RULES_1

out, metrics = run_single_event(
    "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv",
    model="gpt-5-nano",
    rules=RULES_1,
    out_root="runs",
    tag="modeR-gpt-5-nano-RULES1-probe",
    max_output_tokens=1000,
    max_rows=None
)

Event: kaikoura_earthquake_2016
Labels in scope: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort', 'sympathy_and_support', 'other_relevant_information', 'not_humanitarian']
Processing 435 rows with up to 5 attempts per row...
Processed 20/435 rows... (success rate: 100.0%)
Processed 40/435 rows... (success rate: 100.0%)
Processed 60/435 rows... (success rate: 100.0%)
Processed 80/435 rows... (success rate: 100.0%)
Processed 100/435 rows... (success rate: 100.0%)
Processed 120/435 rows... (success rate: 100.0%)
Processed 140/435 rows... (success rate: 100.0%)
Processed 160/435 rows... (success rate: 100.0%)
Processed 180/435 rows... (success rate: 100.0%)
Processed 200/435 rows... (success rate: 100.0%)
Processed 220/435 rows... (success rate: 100.0%)
Processed 240/435 rows... (success rate: 100.0%)
Processed 260/435 rows... (success rate: 100.0%)

In [2]:
from rules import RULES_1

out, metrics = run_single_event(
    "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv",
    model="gpt-5-mini",
    rules=RULES_1,
    out_root="runs",
    tag="modeR-gpt-5-mini-RULES1-probe",
    max_output_tokens=1000,
    max_rows=None
)

Event: kaikoura_earthquake_2016
Labels in scope: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort', 'sympathy_and_support', 'other_relevant_information', 'not_humanitarian']
Processing 435 rows with up to 5 attempts per row...
Processed 20/435 rows... (success rate: 100.0%)
Processed 40/435 rows... (success rate: 100.0%)
Processed 60/435 rows... (success rate: 100.0%)
Processed 80/435 rows... (success rate: 100.0%)
Processed 100/435 rows... (success rate: 100.0%)
Processed 120/435 rows... (success rate: 100.0%)
Processed 140/435 rows... (success rate: 100.0%)
Processed 160/435 rows... (success rate: 100.0%)
Processed 180/435 rows... (success rate: 100.0%)
Processed 200/435 rows... (success rate: 100.0%)
Processed 220/435 rows... (success rate: 100.0%)
Processed 240/435 rows... (success rate: 100.0%)
Processed 260/435 rows... (success rate: 100.0%)

In [2]:
from rules import RULES_1

out, metrics = run_single_event(
    "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv",
    model="gpt-5-mini",
    rules=RULES_1,
    out_root="runs",
    tag="modeR-gpt-5-mini-RULES1-probe",
    max_output_tokens=50,
    max_rows=None
)

Event: kaikoura_earthquake_2016
Labels in scope: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort', 'sympathy_and_support', 'other_relevant_information', 'not_humanitarian']
Processing 435 rows with up to 5 attempts per row...
  ⚠ Invalid label 'None' on attempt 1/5, retrying...
  ⚠ Invalid label 'None' on attempt 2/5, retrying...
  ⚠ Invalid label 'None' on attempt 3/5, retrying...
  ⚠ Invalid label 'None' on attempt 4/5, retrying...
  ✗ Failed to get valid label after 5 attempts (got: 'None')
    → Using fallback: 'not_humanitarian'
  ⚠ Invalid label 'None' on attempt 1/5, retrying...
  ⚠ Invalid label 'None' on attempt 2/5, retrying...
  ⚠ Invalid label 'None' on attempt 3/5, retrying...
  ⚠ Invalid label 'None' on attempt 4/5, retrying...
  ✗ Failed to get valid label after 5 attempts (got: 'None')
    → Using fallback: 'not_humanitarian'
  ⚠ 

KeyboardInterrupt: 